# 🔭 DepthScope: Monocular Depth Estimation
### Predicting 3D Scene Geometry from a Single 2D RGB Image

Welcome to **DepthScope**! In this hands-on computer vision experiment, we explore **Monocular Depth Estimation (MDE)** using a state-of-the-art pretrained deep learning model.

---

### 🧠 What is Monocular Depth Estimation?
Humans perceive depth and 3D structure through a combination of binocular vision (two eyes providing stereo disparity) and monocular visual cues. Even when looking at a single 2D photograph with one eye closed, you can intuitively discern what is close and what is far away by recognizing:
- **Perspective and vanishing lines**: Converging parallel lines signal receding distance.
- **Occlusion**: Foreground objects visually overlap and block background objects.
- **Texture gradients**: Surface textures appear denser and finer as distance increases.
- **Relative size**: Known objects (people, cars, trees) subtend smaller visual angles when situated farther away.

**Monocular Depth Estimation (MDE)** is the computer vision task of predicting a dense depth map from a single 2D RGB photograph. Rather than requiring multi-view stereo rigs, LiDAR sensors, or Time-of-Flight (ToF) cameras, MDE relies entirely on learning these visual cues from vast image datasets.

---

### 🤖 The Pretrained Model: Depth Anything V2 (Small)
In this experiment, we utilize **Depth Anything V2 Small** (`depth-anything/Depth-Anything-V2-Small-hf`), developed by ByteDance and HKU:
- **Vision Transformer (ViT) Backbone**: Extracts multi-scale visual tokens and reasons over long-range spatial context.
- **Zero-Shot Robustness**: Trained on massive synthetic and unlabeled real-world datasets, enabling it to generalize to diverse indoor and outdoor scenes.
- **Lightweight & Fast**: The `Small` checkpoint delivers high-detail relative depth predictions with fast inference on Google Colab GPUs (and smooth fallback on CPU).
- **Relative Depth Output**: Predicts per-pixel relative disparity (closer vs. farther) across the entire scene.

Let's step through the pipeline! 🚀

In [ ]:
# install transformers and accelerate quietly so colab stays clean
!pip install -q transformers accelerate

In [ ]:
# pull in all the standard tools we need
import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline

# check colab environment for upload/download utils
try:
    from google.colab import files
    COLAB_ENV = True
except ImportError:
    COLAB_ENV = False
    print("ℹ️ Running outside Colab environment - local file paths will be used.")

# grab cuda if we got it, else chill on cpu
device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"⚡ Compute device: {device_name} (device id: {device})")

In [ ]:
# load up depth anything v2 small once so we don't reload every time
model_id = "depth-anything/Depth-Anything-V2-Small-hf"

print(f"📦 Loading pretrained depth estimation model: {model_id}...")

# instantiate hugging face depth-estimation pipeline
pipe = pipeline(
    task="depth-estimation",
    model=model_id,
    device=device
)

print("✅ Pipeline loaded and ready to cook!")

In [ ]:
# prompt user to upload a single photo
uploaded_image_path = None

if COLAB_ENV:
    print("📁 Please upload a photograph (supported formats: JPG, JPEG, PNG)...")
    uploaded = files.upload()
    
    # make sure user actually dropped an image
    if not uploaded:
        raise ValueError("❌ No file uploaded! Please re-run this cell and choose a valid image file.")
    
    # grab the first uploaded file name
    uploaded_file_name = list(uploaded.keys())[0]
    
    # validate file extension
    valid_exts = (".jpg", ".jpeg", ".png")
    if not uploaded_file_name.lower().endswith(valid_exts):
        raise ValueError(f"❌ Unsupported file format for '{uploaded_file_name}'. Supported formats: JPG, JPEG, PNG.")
        
    uploaded_image_path = uploaded_file_name
    print(f"🎉 Upload successful: {uploaded_image_path}")
else:
    # local testing fallback
    sample_path = "sample.jpg"
    if os.path.exists(sample_path):
        uploaded_image_path = sample_path
        print(f"📁 Using local fallback image: {uploaded_image_path}")
    else:
        raise FileNotFoundError("❌ No image found. In Colab, run files.upload(). Locally, place 'sample.jpg' in the directory.")

In [ ]:
# open image with pil and force rgb so no weird channels
try:
    raw_image = Image.open(uploaded_image_path)
    img_rgb = raw_image.convert("RGB")
    
    # grab and display image dimensions
    img_width, img_height = img_rgb.size
    print(f"🖼️ Image successfully opened!")
    print(f"📐 Dimensions: {img_width} x {img_height} pixels (Width x Height)")
    print(f"🎨 Image Color Mode: {img_rgb.mode}")
except Exception as e:
    raise RuntimeError(f"❌ Could not open or process image '{uploaded_image_path}': {e}")

In [ ]:
# clock how fast the model cooks
start_time = time.perf_counter()

try:
    # run inference through the depth pipeline without extra fine-tuning
    with torch.no_grad():
        output = pipe(img_rgb)
    end_time = time.perf_counter()
    
    inference_time_ms = (end_time - start_time) * 1000.0
    print(f"⚡ Inference completed in {inference_time_ms:.2f} ms ({end_time - start_time:.4f} seconds)")
    print(f"📊 Original input size: {img_width} x {img_height}")
except Exception as e:
    raise RuntimeError(f"❌ Depth inference failed: {e}")

In [ ]:
# safely extract depth and normalize to [0, 1]
# check if raw predicted_depth tensor is provided
if "predicted_depth" in output and output["predicted_depth"] is not None:
    raw_tensor = output["predicted_depth"]
    
    # move tensor to cpu and strip extra batch dims
    if hasattr(raw_tensor, "detach"):
        raw_depth_np = raw_tensor.detach().cpu().numpy().squeeze()
    else:
        raw_depth_np = np.array(raw_tensor).squeeze()
        
    # safe min-max normalization to dodge division by zero
    d_min = float(raw_depth_np.min())
    d_max = float(raw_depth_np.max())
    
    if d_max - d_min > 1e-6:
        depth_normalized = (raw_depth_np - d_min) / (d_max - d_min)
    else:
        depth_normalized = np.zeros_like(raw_depth_np, dtype=np.float32)
        
    # resize normalized depth map back to match original image size for crisp display
    depth_map_pil = Image.fromarray((depth_normalized * 255.0).astype(np.uint8)).resize(
        img_rgb.size, resample=Image.Resampling.BILINEAR
    )
    depth_display = np.array(depth_map_pil) / 255.0
else:
    # fallback to the pre-rendered pil depth image
    depth_map_pil = output["depth"].convert("L")
    depth_display = np.array(depth_map_pil, dtype=np.float32) / 255.0

# plot side-by-side: original, grayscale depth, colorized depth map
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

# 1. original photograph
axes[0].imshow(img_rgb)
axes[0].set_title(f"1. Original Photograph\n({img_width} x {img_height})", fontsize=13, fontweight="bold")
axes[0].axis("off")

# 2. grayscale relative depth map
axes[1].imshow(depth_display, cmap="gray")
axes[1].set_title("2. Grayscale Depth Map\n(Brighter = Closer, Darker = Farther)", fontsize=13, fontweight="bold")
axes[1].axis("off")

# 3. colorized relative depth map with colorbar
im_color = axes[2].imshow(depth_display, cmap="inferno")
axes[2].set_title("3. Colorized Relative Depth Map\n(Inferno Colormap)", fontsize=13, fontweight="bold")
axes[2].axis("off")

# add colorbar to show relative scale
cbar = fig.colorbar(im_color, ax=axes[2], fraction=0.046, pad=0.04)
cbar.set_label("Relative Disparity / Depth (Normalized [0, 1])", fontsize=11)

plt.show()

print("💡 Note: Relative depth values represent normalized disparity (closer vs farther), not physical metres or centimetres.")

In [ ]:
# save the generated depth map as a png
output_filename = "depthscope_predicted_depth.png"

# save grayscale depth map using pillow
depth_map_pil.save(output_filename, format="PNG")
print(f"💾 Depth map saved locally as '{output_filename}'")

# trigger browser download if running in colab
if COLAB_ENV:
    try:
        print("⬇️ Triggering browser download...")
        files.download(output_filename)
        print("✅ Download started!")
    except Exception as e:
        print(f"⚠️ Automatic download prompt skipped or blocked: {e}")
        print(f"You can access '{output_filename}' directly in the Colab files panel on the left.")
else:
    print(f"✅ Saved depth map to: {os.path.abspath(output_filename)}")

## 🔬 Experiment Analysis & Conceptual Guide

Now that you have seen the depth map produced by Depth Anything V2, let's break down the science behind what the model is actually doing and where monocular vision meets its limits.

---

### 1. What Does the Depth Map Represent?
- In monocular depth models like **Depth Anything V2**, the predicted value for each pixel represents **relative inverse depth (disparity)**.
- In our normalized visualization (`[0, 1]`):
  - **High values (Brighter / Warm yellow in Inferno colormap)** represent objects that are **closer** to the camera.
  - **Low values (Darker / Cool purple/black in Inferno colormap)** represent objects that are **farther** away in the background.
- This produces a dense ordering of scene elements from foreground to background.

---

### 2. Relative Depth vs. Metric Depth
It is critical to distinguish between **relative depth** and **metric depth**:
| Property | Relative Depth (DepthScope) | Metric Depth |
| :--- | :--- | :--- |
| **Units** | Unitless normalized scale (`[0, 1]`) | Physical units (meters, centimeters, feet) |
| **Output** | Affine-invariant / Disparity ordering | True Euclidean distance ($Z$) from camera plane |
| **Requirements** | Single RGB photograph only | Camera focal length, sensor size, baseline, or LiDAR calibration |
| **Scale Ambiguity** | Inherent: a small toy car close up looks identical to a real car far away | Resolved: known scale and physical distance |
| **Use Cases** | Bokeh blur effects, 3D photography, AR relighting, obstacle prioritization | Robotic grasping, autonomous vehicle navigation, industrial metrology |

> **Scale Ambiguity**: From a single 2D photo without camera calibration, it is mathematically impossible to know whether you are viewing a miniature model 30 cm away or a full-scale building 300 meters away. Relative depth models predict the *ordering* and *shape*, but cannot give real-world meter measurements.

---

### 3. Failure Cases: Transparent and Reflective Surfaces
Monocular depth networks are trained on visual texture and diffuse surface reflection assumptions. When these assumptions fail, depth predictions can distort:
- **Glass and Transparent Windows**: Glass surfaces do not reflect diffuse light in the standard way. The model often sees right through the window and predicts the depth of whatever is behind the glass, or gets confused by faint reflections on the surface.
- **Mirrors and Specular Reflections**: A mirror reflects the distant room or objects. Because the model relies on appearance cues, it often predicts depth *inside* the virtual image of the mirror (assigning it far depth) rather than detecting the flat surface of the mirror itself.
- **Polished Floors and Water**: Wet pavement, marble floors, or water bodies produce mirror-like specular highlights that violate standard Lambertian surface assumptions.

---

### 4. General Limitations of Monocular Depth Estimation
1. **Ill-Posed Mathematical Problem**: Projecting a 3D world onto a 2D plane loses the third dimension ($Z$). Multiple distinct 3D shapes can produce the exact same 2D projection.
2. **Thin Structures & Occlusion Boundaries**: Fine details like power lines, hair strands, bicycle spokes, and tree branches may blur or merge into the background.
3. **Lighting & Shadow Dependence**: Deep shadows can be mistaken for distant voids, while strong spotlight glare can be mistaken for nearby high-frequency surfaces.
4. **Out-of-Distribution Scenes**: Highly unusual perspectives (e.g., extreme drone nadir shots, microscopic imagery, or abstract art) may not match the distribution of the training data.

## 📝 Results & Observations Log

Use the template below to record your findings as you experiment with different photographs in DepthScope.

---

### 🧪 Test 1: Indoor Scene (e.g., Desk, Living Room, Hallway)
- **Image Description**:
- **Inference Time (ms)**:
- **Key Observations**:
  - How well were foreground objects (e.g., mug, keyboard, chair) separated from the background?
  - Did edge boundaries appear sharp or blurry?
  - Did indoor ceiling lights or shadows create false depth artifacts?

---

### 🌲 Test 2: Outdoor Scene (e.g., Street, Park, Architecture)
- **Image Description**:
- **Inference Time (ms)**:
- **Key Observations**:
  - How did the model handle the sky? (The sky should have very low disparity / farthest distance).
  - How well did perspective cues (roads, sidewalks, rows of trees) translate into smooth depth gradients?
  - Were distant buildings properly separated from foreground trees or pedestrians?

---

### 🪞 Test 3: Reflective or Transparent Object (e.g., Mirror, Glass Window, Water)
- **Image Description**:
- **Inference Time (ms)**:
- **Key Observations**:
  - Did the model predict the depth of the glass surface or the objects behind the glass?
  - For mirrors, did the depth map depict the wall where the mirror hangs or the reflected room inside?
  - How did specular reflections affect nearby surface predictions?

---

### 📌 Summary of Personal Takeaways
*(Record your reflections on the performance of Depth Anything V2 Small, speed vs. accuracy trade-offs, and where you would apply relative depth maps in real-world creative or technical projects.)*